# UAV

## CSF-FPE Transformer（Continuous Spectral Fusion with Positional Encoding）

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

DATA_DIR = r"\data\input"
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
model_save_dir = os.path.join(DATA_DIR, "UAVtoS2_ablation/CSF-FPE")
os.makedirs(model_save_dir, exist_ok=True)

# =========================
# ⭐ Dataset selection
# =========================
selected_datasets = ["uav2s2"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
    "uav2s2": (3, "UAV_Sentinel_S2A.csv"),
    "uav2l8": (4, "UAV_Landsat8_SRF.csv")
}

# =========================
# 0. Spectral normalization settings
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1 or dtype == 3:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 2:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(
            self,
            d_model=64,
            grid_size=64,

            use_projection=True,
            use_fourier_positional_encoding=True,
            use_temp_token=True
    ):
        super().__init__()
        # =========================
        # 1. Ablation switches
        # =========================
        self.use_projection = use_projection
        self.use_fourier_positional_encoding = use_fourier_positional_encoding
        self.use_temp_token = use_temp_token

        #=========================
        # 2. Attention dimensions
        #=========================
        self.d_model = d_model

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4.1 Spectral grid (fixed coordinate system)
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )
        # =========================
        # 4.2 Learnable spectral diffusion weights
        # =========================
        self.spectral_weight_net = nn.Sequential(
            nn.Linear(4, 32),
            nn.GELU(),
            nn.Linear(32, 16),
            nn.GELU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

        #=========================
        # replace4. MLP for spectral projection
        #=========================
        self.project_mlp = nn.Sequential(
            nn.Linear(d_model,d_model),
            nn.ReLU(),
            nn.Linear(d_model,d_model)
        )

        # =========================
        # 5. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        #=========================
        # replace5. MLP for positional encoding
        #=========================
        self.pos_mlp = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        #=========================
        # 6. Feature fusion layer
        #=========================
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 7. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        # replace7. MLP for Encoder
        # =========================
        self.encoder_mlp = nn.Sequential(
            nn.Linear(d_model,2*d_model),
            nn.GELU(),
            nn.Linear(2*d_model,d_model),
            nn.GELU()
        )

        # =========================
        # 7. Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)
    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)
    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. spectral projection
        # -------------------------
        if self.use_projection:
            z = self.spectral_project(
                z,
                wl.squeeze(-1),
                delta.squeeze(-1)
            )

        else:
            z = F.interpolate(
                z.transpose(1,2),
                size=len(self.grid),
                mode="linear",
                align_corners=False
            ).transpose(1,2)

            z = self.project_mlp(z)

        # -------------------------
        # 3. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        if self.use_fourier_positional_encoding:
            pos = self.fourier_encoding(grid)
            z = torch.cat([z,pos],dim=-1)
            z = self.fusion(z)

        else:
            pos = self.pos_mlp(grid.unsqueeze(-1))
            z = z + pos

        # -------------------------
        # 4. Encoder
        # -------------------------
        B = z.size(0)

        if self.use_temp_token:
            temp = temp
        else:
            temp = torch.rand_like(temp)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t,  z], dim=1)          # (B,N+1,d)

        # -------------------------
        # 5. CLS token pooling
        # -------------------------
        if self.use_fourier_positional_encoding:
            z = self.encoder(z)
            z = z[:, 0]  # CLS token
        else:
            z = self.encoder_mlp(z)
            z = z.mean(dim=1)

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2

In [ ]:
import os
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from itertools import combinations

# =========================
# Ablation configuration generator
# =========================
MODULES = ["projection", "fourier_positional_encoding", "temp_token"]

def generate_ablation_configs():
    configs = {}

    # Full model
    configs["Full"] = {
        "use_projection": True,
        "use_fourier_positional_encoding": True,
        "use_temp_token": True
    }

    # all combinations
    for r in range(1, len(MODULES) + 1):
        for comb in combinations(MODULES, r):

            name = "No_" + "_".join(comb)

            cfg = {
                "use_projection": True,
                "use_fourier_positional_encoding": True,
                "use_temp_token": True
            }

            for m in comb:
                cfg[f"use_{m}"] = False

            configs[name] = cfg

    return configs


ABLATION_CONFIG = generate_ablation_configs()

# =========================
# train function
# =========================
def train_one_experiment(exp_name, cfg):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n" + "=" * 80)
    print(f"Running Experiment : {exp_name}")
    print("=" * 80)

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # normalization
    # =========================
    all_x = []
    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)

    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    dataset = SpectralDataset(
        paths[0][1],
        paths[0][0],
        x_mean,
        x_std
    )

    folds = json.load(open(SPLIT_PATH))

    results = []
    fold_records = []   # ⭐ NEW: store fold-level metrics

    # =========================
    # 5-fold training
    # =========================
    for fold_id in range(5):

        print(f"\n==== {exp_name} Fold {fold_id} ====")

        train_idx = np.array(folds[fold_id]["train_idx"])
        test_idx = np.array(folds[fold_id]["test_idx"])

        train_loader = DataLoader(
            Subset(dataset, train_idx),
            batch_size=16,
            shuffle=True
        )

        test_loader = DataLoader(
            Subset(dataset, test_idx),
            batch_size=16
        )

        model = SpectralFusionModel(**cfg).to(device)

        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        # =========================
        # training loop
        # =========================
        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:

                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"RMSE {rmse:.4f} | R2 {r2:.4f}"
            )

            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # final evaluation
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE : {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE : {mae:.4f}")
        print(f"R2  : {r2:.4f}")

        results.append([mse, rmse, mae, r2])

        # ⭐ NEW: save fold-level record
        fold_records.append({
            "experiment": exp_name,
            "fold": fold_id,
            "mse": mse,
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        })

    # =========================
    # summary
    # =========================
    results = np.array(results)

    print("\n" + "=" * 80)
    print(exp_name)
    print("=" * 80)
    print("FINAL 5-FOLD RESULT")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

    return {
        "mean": results.mean(axis=0),
        "folds": results,
        "records": fold_records   # ⭐ NEW
    }


# =========================
# seed
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# =========================
# main
# =========================
if __name__ == "__main__":

    set_seed(42)

    final_results = {}
    all_fold_results = []   # ⭐ NEW global storage

    for exp_name, cfg in ABLATION_CONFIG.items():

        result = train_one_experiment(exp_name, cfg)

        final_results[exp_name] = result["mean"]

        # ⭐ collect fold-level results
        all_fold_results.extend(result["records"])

    # =========================
    # print summary table
    # =========================
    print("\n" + "=" * 100)
    print("FINAL ABLATION RESULTS")
    print("=" * 100)

    print(f"{'Experiment':<25}{'MSE':<12}{'RMSE':<12}{'MAE':<12}{'R2':<12}")

    for k, v in final_results.items():
        print(
            f"{k:<25}"
            f"{v[0]:<12.4f}"
            f"{v[1]:<12.4f}"
            f"{v[2]:<12.4f}"
            f"{v[3]:<12.4f}"
        )

    # =========================
    # save mean results
    # =========================
    ablation_df = pd.DataFrame(
        final_results,
        index=["MSE", "RMSE", "MAE", "R2"]
    ).T

    ablation_df.to_csv(
        os.path.join(
            DATA_DIR,
            "UAVtoS2_ablation",
            "CSF-FPE",
            "ablation_results.csv"
        )
    )

    # =========================
    # ⭐ SAVE FOLD-LEVEL RESULTS (IMPORTANT)
    # =========================
    fold_df = pd.DataFrame(all_fold_results)

    fold_df.to_csv(
        os.path.join(
            DATA_DIR,
            "UAVtoS2_ablation",
            "CSF-FPE",
            "ablation_fold_results.csv"
        ),
        index=False
    )

    print("\nSaved:")
    print("✔ ablation_results.csv")
    print("✔ ablation_fold_results.csv (fold-level, for plots)")


Running Experiment : Full

==== Full Fold 0 ====
Epoch 000 | TrainLoss 1136.1600 | RMSE 5.1164 | R2 -9.0514
Epoch 001 | TrainLoss 659.6614 | RMSE 3.6598 | R2 -4.1428
Epoch 002 | TrainLoss 323.2902 | RMSE 2.2557 | R2 -0.9536
Epoch 003 | TrainLoss 133.2503 | RMSE 1.4472 | R2 0.1958
Epoch 004 | TrainLoss 66.7301 | RMSE 1.4449 | R2 0.1984
Epoch 005 | TrainLoss 56.9159 | RMSE 1.3941 | R2 0.2538
Epoch 006 | TrainLoss 54.2624 | RMSE 1.4264 | R2 0.2188
Epoch 007 | TrainLoss 53.0287 | RMSE 1.4183 | R2 0.2276
Epoch 008 | TrainLoss 51.6223 | RMSE 1.3467 | R2 0.3036
Epoch 009 | TrainLoss 49.0199 | RMSE 1.4174 | R2 0.2287
Epoch 010 | TrainLoss 47.0331 | RMSE 1.4450 | R2 0.1983
Epoch 011 | TrainLoss 49.5611 | RMSE 1.4775 | R2 0.1618
Epoch 012 | TrainLoss 46.0101 | RMSE 1.4418 | R2 0.2018
Epoch 013 | TrainLoss 45.8286 | RMSE 1.2214 | R2 0.4272
Epoch 014 | TrainLoss 42.1830 | RMSE 1.3067 | R2 0.3443
Epoch 015 | TrainLoss 42.6438 | RMSE 1.6444 | R2 -0.0383
Epoch 016 | TrainLoss 40.6935 | RMSE 1.0403 |